In [ ]:

import kagglehub
sharumaan_semimages_299x299_path = kagglehub.dataset_download('sharumaan/semimages-299x299')

print('Data source import complete.')


# Automated image classification of defects in SEM images

In [ ]:
!pip install natsort --quiet
# !conda install -y natsort

In [ ]:
import os
from os.path import expanduser
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import shutil
import logging
import datetime
import seaborn as sns
from collections import Counter
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import keras
from tensorflow.keras.utils import to_categorical # used for converting labels to one-hot-encoding (這邊修改過，改成新的版本)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf

In [ ]:
base_url = "https://b2share.eudat.eu/records/qtq9v-ys023/files"

classes = [
    "Biological",
    "Fibres",
    "Films_Coated_Surface",
    "MEMS_devices_and_electrodes",
    "Nanowires",
    "Particles",
    "Patterned_surface",
    "Porous_Sponge",
    "Powder",
    "Tips",
]

for cname in classes:
    url = f"{base_url}/{cname}.tar?download=1"
    out_name = f"{cname}.tar"
    print("Downloading:", url)
    !wget -O "$out_name" "$url"


In [ ]:
import os
import tarfile

extract_root = "/content/sem_dataset"
os.makedirs(extract_root, exist_ok=True)

classes = [
    "Biological",
    "Fibres",
    "Films_Coated_Surface",
    "MEMS_devices_and_electrodes",
    "Nanowires",
    "Particles",
    "Patterned_surface",
    "Porous_Sponge",
    "Powder",
    "Tips",
]

for cname in classes:
    tar_path = f"{cname}.tar"
    print("Extracting:", tar_path)
    with tarfile.open(tar_path, "r") as tar:
        tar.extractall(path=extract_root)

!ls -R /content/sem_dataset


檢查一下資料結構

In [ ]:
import os

data_root = "/content/sem_dataset"

for cname in sorted(os.listdir(data_root)):
    cpath = os.path.join(data_root, cname)
    if os.path.isdir(cpath):
        n_imgs = len([f for f in os.listdir(cpath)
                      if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"))])
        print(f"{cname:30s} -> {n_imgs} images")


切 train / val / test（實際複製檔案到新資料夾）

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

def collect_image_paths(root_dir, exts=IMAGE_EXTS):
    class_names = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])

    paths, labels = [], []
    for cname in class_names:
        cdir = os.path.join(root_dir, cname)
        for fname in os.listdir(cdir):
            if fname.lower().endswith(exts):
                paths.append(os.path.join(cdir, fname))
                labels.append(cname)

    print(f"Found {len(paths)} images in {len(class_names)} classes:")
    print(class_names)
    return paths, labels, class_names


def make_splits(root_dir, out_dir,
                test_size=0.15, val_size=0.15,
                random_state=42):
    paths, labels, class_names = collect_image_paths(root_dir)

    # 切 test
    X_temp, X_test, y_temp, y_test = train_test_split(
        paths, labels,
        test_size=test_size,
        stratify=labels,
        random_state=random_state
    )

    # 切 val（剩下的）
    val_ratio_relative = val_size / (1.0 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=val_ratio_relative,
        stratify=y_temp,
        random_state=random_state
    )

    def _copy_split(X, y, split_name):
        for src, label in zip(X, y):
            dst_dir = os.path.join(out_dir, split_name, label)
            os.makedirs(dst_dir, exist_ok=True)
            dst = os.path.join(dst_dir, os.path.basename(src))
            shutil.copy2(src, dst)

    _copy_split(X_train, y_train, "train")
    _copy_split(X_val,   y_val,   "val")
    _copy_split(X_test,  y_test,  "test")

    print("Done splitting.")
    print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

    return {
        "class_names": class_names,
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }

root_dir = "/content/sem_dataset"
out_dir  = "/content/sem_dataset_splits"

splits_info = make_splits(root_dir, out_dir,
                          test_size=0.15,
                          val_size=0.15,
                          random_state=42)

!ls -R /content/sem_dataset_splits


建立 ImageDataGenerator（資料增強 + dataloader）

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

data_root = "/content/sem_dataset_splits"
img_size = (299, 299)   # InceptionV3 預設輸入尺寸
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)  # 明確分開，不共用 val_datagen

train_gen = train_datagen.flow_from_directory(
    directory=os.path.join(data_root, "train"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    directory=os.path.join(data_root, "val"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical"
)

test_gen = test_datagen.flow_from_directory(
    directory=os.path.join(data_root, "test"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)

# 全域共用，供後續所有 cell 使用，不再重複定義
idx_to_class = {v: k for k, v in train_gen.class_indices.items()}


建立 InceptionV3 transfer learning 模型

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import InceptionV3

num_classes = train_gen.num_classes  # 10
input_shape = (299, 299, 3)

base_model = InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=input_shape
)

# 先凍結 backbone
base_model.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=outputs)

model.summary()


compile + 訓練（第一階段：只訓練最後幾層）

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Fix 1: class_weight 解決 Fibres/Porous_Sponge/Films 等少數類別 recall 極低的問題
y_train_labels = train_gen.classes
cw = compute_class_weight("balanced",
                          classes=np.unique(y_train_labels),
                          y=y_train_labels)
class_weights = dict(enumerate(cw))
print("Class weights:", class_weights)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Fix 2: 改用 .keras 格式（.h5 已是 legacy）
checkpoint = ModelCheckpoint(
    "best_sem_cnn.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

epochs = 20  # Fix 4: 從 5 提高到 20，有 EarlyStopping 保護不會過擬合

# Fix 3: steps_per_epoch/validation_steps 使用完整資料集
# 原本硬編碼 50 steps，只跑了 13003/(32*50) ≈ 12% 的訓練資料
history = model.fit(
    train_gen,
    epochs=epochs,
    steps_per_epoch=len(train_gen),
    validation_data=val_gen,
    validation_steps=len(val_gen),
    callbacks=[checkpoint, early_stopping, reduce_lr],
    class_weight=class_weights
)


## Phase 2：Fine-tuning（解凍 InceptionV3 後段 + 更小 LR）
先只訓練 head 讓分類器穩定後，再 unfreeze 最後 50 層做細部調整。

In [ ]:
# Phase 2: unfreeze InceptionV3 最後 50 層進行 fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),  # lr 比 Phase 1 小 100 倍，避免破壞預訓練權重
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

checkpoint_ft = ModelCheckpoint(
    "best_sem_cnn_finetuned.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping_ft = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

history_ft = model.fit(
    train_gen,
    epochs=20,
    steps_per_epoch=len(train_gen),
    validation_data=val_gen,
    validation_steps=len(val_gen),
    callbacks=[checkpoint_ft, early_stopping_ft, reduce_lr],
    class_weight=class_weights
)


跑太久了，調整 batch size

這一輪是「凍結 InceptionV3，只訓練你自己加上去的分類 head」。

在 test set 上評估

In [ ]:
test_loss, test_acc = model.evaluate(test_gen)
print(f"Test loss = {test_loss:.4f}, Test accuracy = {test_acc:.4f}")


混淆矩陣 + classification report

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# 真實標籤
y_true = test_gen.classes
class_indices = test_gen.class_indices  # {'Biological':0, ...}
# idx_to_class 已在 DataGenerator cell 全域定義

# 預測
y_pred_prob = model.predict(test_gen)
y_pred = np.argmax(y_pred_prob, axis=1)

# 混淆矩陣
cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix (raw counts):")
print(cm)

# precision / recall / f1-score
target_names = [idx_to_class[i] for i in range(len(idx_to_class))]
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=target_names))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - SEM 10-class")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 從 test_gen 拿一個 batch
x_batch, y_batch = next(test_gen)
print("Batch shape:", x_batch.shape)

# 2. 取 batch 中的第 0 張
img = x_batch[0]
true_label_idx = np.argmax(y_batch[0])

# 3. 預測
pred_prob = model.predict(np.expand_dims(img, axis=0))[0]
pred_idx = np.argmax(pred_prob)

# 4. 把 index 轉回類別名稱
# idx_to_class 已在 DataGenerator cell 全域定義
true_label = idx_to_class[true_label_idx]
pred_label = idx_to_class[pred_idx]

print("True label:     ", true_label)
print("Predicted label:", pred_label)
print("Class probabilities:")
for i, p in enumerate(pred_prob):
    print(f"  {idx_to_class[i]:30s}: {p:.4f}")

# 5. 把這張圖畫出來看看
plt.imshow(img)      # 已經是 rescale 後的圖片
plt.axis("off")
plt.title(f"True: {true_label}\nPred: {pred_label}")
plt.show()


In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np
import matplotlib.pyplot as plt
import os

img_size = (299, 299)
img_path = "/content/sample_data/100-100-6.5-500x.jpg"

# 1. 讀圖 + resize
img = load_img(img_path, target_size=img_size)
img_arr = img_to_array(img)           # shape: (H, W, 3)
img_arr = img_arr / 255.0             # 跟 ImageDataGenerator 一樣 rescale
img_arr = np.expand_dims(img_arr, 0)  # shape: (1, H, W, 3)

# 2. 預測
pred_prob = model.predict(img_arr)[0]
pred_idx = np.argmax(pred_prob)

# 3. 轉回類別名稱
# idx_to_class 已在 DataGenerator cell 全域定義
pred_label = idx_to_class[pred_idx]

print("Image path:", img_path)
print("Predicted label:", pred_label)
print("Class probabilities:")
for i, p in enumerate(pred_prob):
    print(f"  {idx_to_class[i]:30s}: {p:.4f}")

# 4. 顯示影像
plt.imshow(load_img(img_path))
plt.axis("off")
plt.title(f"Pred: {pred_label}")
plt.show()
